# **5~6주차 : Multi-layer Perceptron (PyTorch)**

In [ ]:
!pip install torch
!pip install tensorflow

### **데이터 불러오기 및 전처리**

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
#from keras.datasets.mnist import load_data ??
#from tensorflow.keras.datasets.mnist import load_data

Cross-validation을 쓰지 않고 train_test_split

딥러닝은 학습 데이터가 이미 많고, 학습 비용이 너무 큼, 

Early Stopping, Learning Rate Scheduling 같은 기술을 사용하기에 학습 중에 검증이 이뤄짐


In [ ]:
# load MNIST dataset
(X_train, y_train),(X_test, y_test) = load_data()

# train, validataion data split
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)

print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}\n")

print(f"y_train: {y_train.shape}")
print(f"y_val: {y_val.shape}")
print(f"y_test: {y_test.shape}")

In [ ]:
# train data visualization
plt.figure(figsize=(15,3))

for i, idx in enumerate(np.random.randint(X_train.shape[0], size=5)):

  img = X_train[idx, ...]
  label = y_train[idx]

  plt.subplot(1,5,i+1)
  plt.imshow(img, cmap="gray")
  plt.title(f'Label: {label}')

데이터들 / 255.0 

픽셀값을 0~255에서 0.0~1.0 범위로 정규화 하는 것

In [ ]:
# data preprocessing
X_train = torch.from_numpy(X_train / 255.0).type(torch.float32)
X_val = torch.from_numpy(X_val / 255.0).type(torch.float32)
X_test = torch.from_numpy(X_test / 255.0).type(torch.float32)

y_train = torch.from_numpy(y_train)
y_val = torch.from_numpy(y_val)
y_test = torch.from_numpy(y_test)

### **모델 생성**

`torch.nn.Module` 을 상속받는 클래스 작성

`__init__` 함수에서 사용할 레이어들을 초기화

`forward` 함수에서 입력 데이터에 대한 연산 구현

In [ ]:
import torch
import torch.nn as nn

<div style="font-size:14px; color:#555555;">
<b>nn.Module이란?</b><br>
PyTorch에서 신경망 모델을 만드는 <b>기본 틀(부모 클래스)</b><br><br>

<code>nn.Module</code>을 상속받아서 내가 원하는 구조의 모델을 정의할 수 있다.
</div>

In [ ]:
# model definition
class MyModel(nn.Module):
    def __init__(self, num_labels):
        super(MyModel, self).__init__()  # 부모 클래스 초기화
        self.flatten = nn.Flatten()  # 2D → 1D 벡터
        self.dense1 = nn.Sequential(nn.Linear(784,392), nn.ReLU())   # 1번째 은닉층 (선형: 784 → 392) (비선형: ReLU)
        self.dense2 = nn.Sequential(nn.Linear(392,198), nn.ReLU())   # 2번째 은닉층 (선형: 392 → 198) (비선형: ReLU)
        self.output = nn.Linear(198, num_labels)  # 출력층: 198 → 10

    def forward(self, x):
        x = self.flatten(x)
        x = self.dense1(x)
        x = self.dense2(x)
        x = self.output(x)
        return x

# model creation
model = MyModel(num_labels=10)

<div style="font-size:14px; color:#555555;">
<b>model(x)를 호출하면 PyTorch 내부에서 자동으로 forward(x)를 실행한다</b><br>

<code>__init__</code>: 레이어를 정의 (모델 생성 시 1회만)


<code>forward</code>: 레이어들을 사용하는 순서를 정의 (모델 호출 할 때마다 실행)
</div>

In [ ]:
# display model structure
print(model)

### **모델 학습**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# hyperparameter setting
epochs = 10  #전체 데이터를 몇 번 반복 학습할지
batch_size = 128   #1번에 처리할 데이터 개수
lr = 1e-4  #학습률
weight_decay = 1e-2 #과적합 방지용 파라미터


# loss function
criterion = nn.CrossEntropyLoss()
#optimizer setting
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

In [ ]:
# dataset, dataloader creation
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

<div style="font-size:14px; color:#555555;">
<b>model(x)를 호출하면 PyTorch 내부에서 자동으로 forward(x)를 실행한다</b><br>

<code>TensorDataset</code>: (입력, 정답) 쌍을 묶는 데이터셋 객체

<code>DataLoader</code>: 데이터를 배치단위로 꺼내주는 객체

<code>shuffle=True</code>: 학습마다 순서를 섞어 과적합 방지

<code>shuffle=False</code>: 검증/테스트는 순서가 중요하지 않아서 섞지 않음
</div>

In [ ]:
# model training
for epoch in range(1,epochs+1):
    train_loss, train_acc, val_loss, val_acc = [], [], [], []
    model.train()                              # training mode of model
                                               #학습 모드로 전환 : 드롭 아웃 활성화
    for data, label in train_dataloader:
        pred = model(data) #Forward: 입력 -> 출력 순서로 계산함
        loss = criterion(pred, label) #loss 계산

        train_loss.append(loss)
        train_acc.append((pred.argmax(dim=-1)==label).sum() / len(label))

        optimizer.zero_grad()                  # remove accumulated gradient of parameters
        loss.backward()                        # gradient accumulation (역전파)
        optimizer.step()                       # parameter update by accumulated gradient and optimizer

    model.eval()                               # evaluation mode of model
    for data, label in val_dataloader:
        with torch.no_grad():                  # turn off autograd (for calculation speed, memory efficiency)
            pred = model(data)
            loss = criterion(pred, label)
            val_loss.append(loss)
            val_acc.append((pred.argmax(dim=-1)==label).sum() / len(label))

<div style="font-size:14px; color:#555555;">
<b>epochs 반복 구간</b><br>
<b> ㄴ 학습 구간 (train_dataloader)</b><br>

(1) forward → (2) loss 계산 → (3) gradient 초기화 → (4) backward → (5) gradient 업데이트

<code>model(data)</code>: (1)<br>
<code>criterion(pred, label)</code>: (2)<br>
<code>optimizer.zero_grad()</code>: (3)<br>
<code>loss.backward</code>: (4)<br>
<code>optimizer.step()</code>: (5)<br>

<b> ㄴ 검증 구간 (val_dataloader)</b><br>
순전파 → 손실/정확도 기록만 (가중치 수정 없음)

<code>model.eval()</code>: 드롭아웃 끄기, Batch Normalization 고정 → 예측 결과를 안정적으로<br>
<code>with torch.no_grad()</code>: 기울기 계산 자체를 끔 → 메모리 절약, 속도 향상<br>
</div>

In [ ]:

    train_loss = torch.stack(train_loss).mean().item()
    train_acc = torch.stack(train_acc).mean().item()
    val_loss = torch.stack(val_loss).mean().item()
    val_acc = torch.stack(val_acc).mean().item()

    print(f"Epoch {epoch}/{epochs}")
    print(f"train loss: {round(train_loss, 4)}  train accuracy: {round(train_acc, 4)}")
    print(f"val loss: {round(val_loss, 4)}  val accuracy: {round(val_acc, 4)}\n")

<div style="font-size:14px; color:#555555;">
<b>torch.stack(train_loss)</b><br>
배치별 loss 텐서들을 하나로 합침<br>
<b>.mean()</b> : 배치 평균 → epoch 전체 대표 loss<br>
<b>.item()</b> : 텐서에서 Python 숫자로 꺼냄 (출력용)


<b>train_loss, train_acc, val_loss, val_acc = [], [], [], []</b><br>
매 epoch마다 기록을 초기화. 이 배치들을 나중에 평균 내서 epoch 전체 성능을 구함.

### **모델 평가**

In [ ]:
# model evaluation
test_dataset = TensorDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model.eval()
#학습이 끝난 모델로 Test 데이터를 평가

correct = 0
for data, label in test_dataloader:
    with torch.no_grad():
        pred = model(data)
        #correct에 맞춘 개수를 누적해서 최종 정확도 계산
        correct += (pred.argmax(dim=1)==label).sum()

test_acc = correct / len(test_dataset)

#전체 test 데이터에 대한 정확도 출력
print(f"test accuracy: {test_acc}")

### **모델 저장 및 복원**

`.pt` 또는 `.pth` 확장자를 사용하여 저장 </br> </br>

`torch.save(model.state_dict(), file_path)`를 통해 모델 가중치 저장하기

`torch.save(model, file_path)` 를 통해 모델 전체(구조 및 가중치) 저장하기 </br> </br>

`model.load_state_dict(torch.load(file_path))`를 통해 모델 가중치 불러오기

`torch.load(file_path)` 를 통해 모델 전체(구조 및 가중치) 불러오기

In [ ]:
import torch

#### 방법 1: 모델 전체 저장 (구조 + 가중치)

In [ ]:
# save model and weights
torch.save(model, "model.pth")
torch.save(model.state_dict(), "model_sd.pth")

#### 방법 2: 가중치(state_dict)만 저장

In [ ]:
# load saved model and weights
model1 = torch.load("model.pth")

model2 = MyModel(num_labels=10)
model2.load_state_dict(torch.load("model_sd.pth", map_location="cpu"))